In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "iga_core").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "paper_tables"
FIGURES_DIR = PROJECT_ROOT / "paper_figures"

from iga_core import (
    make_knots,
    elements_spans,
    quadrature_grid,
    basis_ders_on_quad_grid,
    gauss_legendre,
    L2_projection,
    Mass_Matrix,
    Stiffness_Matrix,
    plot_field_1d,
    sol_plot,
    int_approx,
    NIA,
    plots_splines,
    Diff_coef,
    varying_diff_coeff,
    plot_heatmap,
    plot_surface,
)
import matplotlib.pyplot as plt
import numpy as np
from scipy.sparse import csc_matrix, csr_matrix, linalg as sla
from numpy import linspace, zeros, sin, pi, array, cos, exp, eye, dot
from scipy.sparse import csr_matrix, linalg as sla
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
capacity = 1e-5
Lp       = 3.2e-7            # Electrode thickness [m] (320 nm)
sa       = 1e-4              # Electrode area [mÂ²] (1 cmÂ²)
a_max    = 2.33e4            # Max activity [mol/mÂ³]
c0       = 1
F        = 96485             # Faraday constant [C/mol]

In [ ]:
# --- Spatial domain setup ---
ne     = 8
grid   = linspace(0, 1, ne+1)
p      = 3 # degree  of spline
nders  = 1
knots            = make_knots(grid, p, False) # the knots vector
nbasis           = len(knots)-p-1 # number of basis functions
nelements        = len(grid)-1
spans            = elements_spans(knots, p) # last basis function non-vanishing

In [ ]:
U, W            = gauss_legendre(p)   # gauss legendre rule ------> returns  arrays of size (p+1)

points, weights = quadrature_grid(grid, U, W)
basis           = basis_ders_on_quad_grid( knots, p, points, nders, normalize=False ) # is a tensor of order 4 ( ne X (p+1) X nders X nq)

B       = zeros((nbasis, nbasis))
Stiffness_Matrix(nelements, p, spans, basis, weights, points, B)


A       = zeros((nbasis, nbasis))
Mass_Matrix(nelements, p, spans, basis, weights, points, A)

### *Initial solution*

$c(x,0) = cste$

In [ ]:
U_0x = lambda x: c0
U_0  = L2_projection(knots, p, U_0x)
plot_field_1d(knots, p, U_0 , 'r--')
plt.legend(['$c_{Li^+}(x,0)$'])
plt.show()

In [ ]:
D_ref = 1.76e-15  
data = 'DNN'

# --- Temporal domain setup ---
nb_min           = 1
T_0              = Lp**2/D_ref         # seconds 
T                = nb_min*60/T_0       # [no unit]
dt               = 5e-3             # [no unit]
nt               = int(T / dt)  # number of time points including t=0
dt_r             = dt * T_0

t_vals = np.linspace(0.0, T, nt)
t_phys = t_vals * T_0          # convert to seconds

In [ ]:
# Stage 1: evaluate H_j(t) on the uniform PDE time grid before refinement
jmax = 60
H_max = 0.1  # Prescribed safety constraint, not max(H_j)

e0 = np.zeros(nbasis)
e0[0] = 1.0
nx = 101
y = np.linspace(0.0, 1.0, nx)

M = A + dt * B
lu = sla.splu(csr_matrix(M))

cn_jmax = U_0.copy()
H_jmax = np.zeros(nt)
H_jmax[0] = int_approx(nelements, p, spans, basis, weights, cn_jmax) - cn_jmax[0]

jmax_nd = (jmax * capacity * Lp) / (F * sa * D_ref * a_max)
for n in range(1, nt):
    rhs = np.dot(A, cn_jmax) - dt * jmax_nd * e0
    cn_jmax = lu.solve(rhs)
    c_avg_jmax = int_approx(nelements, p, spans, basis, weights, cn_jmax)
    H_jmax[n] = c_avg_jmax - cn_jmax[0]

crossing_indices = np.flatnonzero(H_jmax >= H_max)
if not crossing_indices.size:
    raise ValueError(
        f"H_max={H_max:.8g} is not reached. "
        f"The largest uniform-grid value is {H_jmax.max():.8g}."
    )

idx_s = int(crossing_indices[0])
t_s_nd = float(t_vals[idx_s])
t_s_phys = float(t_phys[idx_s])
idx_peak = int(np.argmax(H_jmax))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_vals, H_jmax, linewidth=2, label=r"$H_j(t)$ for $j(t)=j_{\max}$")
ax.axhline(H_max, color="tab:red", linestyle="--", linewidth=1.5,
           label=rf"$H_{{\max}}={H_max:.4g}$")
ax.axvline(t_s_nd, color="tab:green", linestyle=":", linewidth=1.5,
           label=rf"first crossing $t_s={t_s_nd:.6g}$")
ax.plot(t_vals[idx_peak], H_jmax[idx_peak], "o", color="tab:orange",
        label=rf"peak $H_j={H_jmax[idx_peak]:.4g}$")
ax.set_xlabel("Nondimensional time")
ax.set_ylabel(r"$H_j(t)=c_{\mathrm{avg}}(t)-c_s(t)$")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

print(f"Uniform grid: nt={nt}, dt={dt:.8g}")
print(f"H_j range: [{H_jmax.min():.8g}, {H_jmax.max():.8g}]")
print(f"First H_max crossing: idx_s={idx_s}, t_s_nd={t_s_nd:.8g}, t_s_phys={t_s_phys:.8g} s")


In [ ]:
# Stage 2: refine the temporal control grid using the observed switching time
ne_t = 16
h = T / ne_t
delta = h / 2

t_right = min(T, t_s_nd + delta)

# Coarse before switching, dense immediately after switching, coarse in the tail.
grid_left = np.linspace(0.0, t_s_nd, 8, endpoint=False)
grid_refine = np.linspace(t_s_nd, t_right, 16, endpoint=False)
grid_right = np.linspace(t_right, T, 8)
grid_t = np.unique(np.concatenate([grid_left, grid_refine, grid_right]))

fig, ax = plt.subplots(figsize=(10, 2))
ax.scatter(grid_t, np.zeros_like(grid_t), s=45)
ax.axvline(t_s_nd, color="tab:red", linestyle="--", label=rf"$t_s={t_s_nd:.6g}$")
ax.set_yticks([])
ax.set_xlabel("Nondimensional time")
ax.set_title("Temporal control grid refined around the observed switching time")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


In [ ]:
# Build the temporal B-spline basis on the refined control grid
p_t = 2
knots_t = make_knots(grid_t, p_t, periodic=False)
nbasis_t = len(knots_t) - p_t - 1

t_norm, Phi_t = plots_splines(knots_t, p_t, nt)
n_params = Phi_t.shape[1]

fig, ax = plt.subplots(figsize=(8, 4))
for i in range(n_params):
    ax.plot(t_norm, Phi_t[:, i])
ax.scatter(grid_t, np.zeros(len(grid_t)), s=20, color="black")
ax.set_xlabel("Nondimensional time")
ax.set_ylabel("Temporal basis")
ax.grid(True, alpha=0.2)
fig.tight_layout()
plt.show()

print("n_params", n_params)
print("nbasis_t", nbasis_t)
print("Phi_t shape", Phi_t.shape)

# Build concentration and health responses for the refined temporal basis.
c_basis = np.zeros((n_params, nt, nx))
H_basis = np.zeros((n_params, nt))

for k in range(n_params):
    cn = np.zeros(nbasis)
    phi_k = Phi_t[:, k]
    j_k = (phi_k * capacity * Lp) / (F * sa * D_ref * a_max)

    for n in range(1, nt):
        rhs = np.dot(A, cn) - dt * j_k[n] * e0
        cn = lu.solve(rhs)

        Q, _ = sol_plot(knots, p, cn, nx=nx)
        c_basis[k, n, :] = Q[:, 0]

        c_avg_k = int_approx(nelements, p, spans, basis, weights, cn)
        H_basis[k, n] = c_avg_k - c_basis[k, n, 0]


In [ ]:
def reconstruct_c_and_H(alpha):

    c = np.zeros((nt, nx))
    c_IC = c0 * np.ones(nx)
    avg_list = []
    T_charged = None
    SOC = [0]
    
    for n in range(nt):
        c[n, :] = c_IC
        for k in range(n_params):
            c[n, :] += alpha[k] * c_basis[k, n, :]
        c_avg = np.trapezoid(c[n, :], y)
        avg_list.append(c_avg)
        if c_avg <= 0.6:            
            T_charged = n * dt_r
            SOC.append(T_charged)
    
    H = np.trapezoid(c, y, axis=1) - c[:, 0]   # H(t) = c_avg(t) - c_s(t)
    return c, H, avg_list, SOC[0]




## Maximal feasible solution

In [ ]:
K_pre, K_mix, K_post = [], [], []

for k in range(n_params):
    tL = knots_t[k]
    tR = knots_t[k + p_t + 1]
    if tR <= t_s_nd:
        K_pre.append(k)
    elif tL >= t_s_nd:
        K_post.append(k)
    else:
        K_mix.append(k)

U = K_mix + K_post
print("K_pre",K_pre)
print("K_mix",K_mix)
print("K_post",K_post)



In [ ]:
alpha_opt = np.zeros(n_params)
alpha_opt[K_pre] = jmax 

#--- indices for constraints ---
idx_B = np.arange(idx_s, nt)      # boundary arc collocation indices (temporarily full tail)
idx_1 = np.arange(0, idx_s+1)     # phase-1 collocation indices

#--- (A) boundary constraint: H(t)=H_max for t>=ts ---
A_H = H_basis[U][:, idx_B].T
b_H = H_max - (H_basis[K_pre][:, idx_B].T @ alpha_opt[K_pre])

#--- (B) phase-1 constraint: j_amp(t)=jmax for t<=ts ---
A_J = Phi_t[idx_1][:, U]
b_J = jmax*np.ones(len(idx_1)) - Phi_t[idx_1][:, K_pre] @ (jmax*np.ones(len(K_pre)))

#--- stack with a large weight to enforce phase-1 strongly ---
lam = 1e4
A_s   = np.vstack([A_H, lam*A_J])
b   = np.concatenate([b_H, lam*b_J])

# alpha_U, *_ = np.linalg.lstsq(A_s, b, rcond=None)
# alpha_opt[U] = alpha_U

# ============================================================
# Solve overdetermined system
# ============================================================

alpha_U, residuals_lstsq, rank_As, svals_As = np.linalg.lstsq(A_s, b, rcond=None)
alpha_opt[U] = alpha_U

# Reconstruct current and health
j_amp = Phi_t @ alpha_opt
H_rec = H_basis.T @ alpha_opt

print("j_amp min/max:", j_amp.min(), j_amp.max())


# ============================================================
# Residual diagnostics
# ============================================================

# Boundary arc residual: r_H = H - Hmax on t >= ts
r_H = H_rec[idx_B] - H_max

# Phase-1 current residual: r_J = j - jmax on t <= ts
r_J = j_amp[idx_1] - jmax

# Weighted algebraic residual of the stacked system
r_stack = A_s @ alpha_U - b

# Unweighted residuals from the two constraints
r_H_alg = A_H @ alpha_U - b_H
r_J_alg = A_J @ alpha_U - b_J

# Norms for boundary health residual
rH_inf = np.linalg.norm(r_H, ord=np.inf)
rH_2   = np.linalg.norm(r_H, ord=2) / np.sqrt(len(r_H))
rH_max_violation = np.max(r_H)
rH_mean_abs = np.mean(np.abs(r_H))

# Norms for phase-1 current residual
rJ_inf = np.linalg.norm(r_J, ord=np.inf)
rJ_2   = np.linalg.norm(r_J, ord=2) / np.sqrt(len(r_J))
rJ_mean_abs = np.mean(np.abs(r_J))

# Algebraic residuals
rH_alg_inf = np.linalg.norm(r_H_alg, ord=np.inf)
rH_alg_2   = np.linalg.norm(r_H_alg, ord=2) / np.sqrt(len(r_H_alg))

rJ_alg_inf = np.linalg.norm(r_J_alg, ord=np.inf)
rJ_alg_2   = np.linalg.norm(r_J_alg, ord=2) / np.sqrt(len(r_J_alg))

r_stack_inf = np.linalg.norm(r_stack, ord=np.inf)
r_stack_2   = np.linalg.norm(r_stack, ord=2) / np.sqrt(len(r_stack))

# Condition number estimate
cond_As = svals_As[0] / svals_As[-1]

print("\n================ Residual diagnostics ================")
print("Boundary arc residual r_H = H - Hmax")
print("  ||r_H||_inf        =", rH_inf)
print("  RMS(r_H)           =", rH_2)
print("  max(H-Hmax)        =", rH_max_violation)
print("  mean |r_H|         =", rH_mean_abs)

print("\nPhase-1 current residual r_J = j - jmax")
print("  ||r_J||_inf        =", rJ_inf)
print("  RMS(r_J)           =", rJ_2)
print("  mean |r_J|         =", rJ_mean_abs)

print("\nAlgebraic residuals before weighting")
print("  ||A_H alpha_U - b_H||_inf =", rH_alg_inf)
print("  RMS(A_H alpha_U - b_H)    =", rH_alg_2)
print("  ||A_J alpha_U - b_J||_inf =", rJ_alg_inf)
print("  RMS(A_J alpha_U - b_J)    =", rJ_alg_2)

print("\nWeighted stacked system")
print("  ||A_s alpha_U - b||_inf =", r_stack_inf)
print("  RMS(A_s alpha_U - b)    =", r_stack_2)
print("  rank(A_s)               =", rank_As)
print("  cond(A_s)               =", cond_As)
print("=======================================================")

j_amp = Phi_t @ alpha_opt
print("j_amp min/max:", j_amp.min(), j_amp.max())

In [ ]:
#--- reconstruct ---
c_opt, H_opt, avg_list, SOC = reconstruct_c_and_H(alpha_opt)


params = {
    "Lp": Lp,
    "c0": c0,
    "a_max": a_max,
    "D_ref": D_ref,
    "H_max": H_max,
    "j_amp": j_amp,
    "T": T
}

np.savez(
    DATA_DIR / "figure_05_linear_solution.npz",
    j_amp=j_amp,
    H_opt=H_opt,
    t_norm=t_norm,
    t_phys=t_phys,
    alpha_opt=alpha_opt,
    params=params
)

print("Refined solution saved successfully!")

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# ============================================================
# Paper-safe Matplotlib settings
# ============================================================
mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "legend.fontsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
})

# Construct a localized admissible control perturbation for panel (d).
# The perturbation is confined to one post-switch temporal B-spline basis
# and is scaled so that the perturbed current remains below j_max.
tol_support = 1e-10
candidate_info = []
for k in K_post:
    support_idx = np.where(Phi_t[:, k] > tol_support)[0]
    if len(support_idx) == 0 or support_idx[0] < idx_s:
        continue
    phi_support = Phi_t[support_idx, k]
    j_support = j_amp[support_idx]
    positive_phi = phi_support > tol_support
    eps_allowed = np.min((jmax - j_support[positive_phi]) / phi_support[positive_phi])
    if eps_allowed > 1e-8:
        center_time = np.mean(t_phys[support_idx])
        candidate_info.append((k, eps_allowed, support_idx, center_time))

if not candidate_info:
    raise RuntimeError("No admissible post-switch perturbation basis was found.")

boundary_mid_time = 0.5 * (t_phys[idx_s] + t_phys[-1])
k_pert, eps_allowed, support_idx, _ = min(
    candidate_info,
    key=lambda item: abs(item[3] - boundary_mid_time),
)
eps = 0.8 * eps_allowed
alpha_pert = alpha_opt.copy()
alpha_pert[k_pert] += eps
j_pert = Phi_t @ alpha_pert
H_pert = H_basis.T @ alpha_pert
ta = t_phys[support_idx[0]]
tb = t_phys[support_idx[-1]]

# ============================================================
# Final paper-style figure (kept as a separate formatting cell)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(7.1, 4.6), constrained_layout=True)

ax = axes[0, 0]
ax.plot(t_phys, j_amp, linewidth=2., label=r"$j^\star(t)$")
ax.axhline(jmax, linestyle="--", linewidth=1.0, color="black", label=r"$j_{\max}$")
ax.set_xlabel("Time [s]")
ax.set_ylabel(r"Charging input $j(t)$")
ax.set_ylim(0, 65)
ax.text(0.02, 0.95, "(a)", transform=ax.transAxes, fontsize=8, fontweight="bold", va="top")
ax.legend(frameon=False, loc="best")

ax = axes[0, 1]
ax.plot(t_phys, H_opt, linewidth=2., label=r"$H^\star(t)$")
ax.axhline(H_max, linestyle="--", linewidth=1.0, color="black", label=r"$H_{\max}$")
ax.set_xlabel("Time [s]")
ax.set_ylabel(r"$H(t)=c_{\mathrm{avg}}(t)-c_s(t)$")
ax.text(0.02, 0.95, "(b)", transform=ax.transAxes, fontsize=8, fontweight="bold", va="top")
ax.legend(frameon=False, loc="best")

ax = axes[1, 0]
ax.plot(t_phys, c_opt[:, 0], linewidth=2., label=r"$c(0,t)$")
ax.plot(t_phys, np.asarray(avg_list), linewidth=2., linestyle="--", label=r"$c_{\mathrm{avg}}(t)$")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Normalized concentration")
ax.text(0.02, 0.95, "(c)", transform=ax.transAxes, fontsize=8, fontweight="bold", va="top")
ax.legend(frameon=False, loc="best")

ax = axes[1, 1]
ax.plot(t_phys, H_opt, linewidth=2., label=r"Nominal $H^\star(t)$")
ax.plot(t_phys, H_pert, linewidth=2., linestyle="--", label=r"Perturbed $\widetilde{H}(t)$")
ax.axhline(H_max, linestyle="--", linewidth=1.0, color="black", label=r"$H_{\max}$")
ax.axvspan(ta + 3, tb + 6, alpha=0.12, label="perturbation support")
ax.set_xlabel("Time [s]")
ax.set_ylabel(r"$H(t)=c_{\mathrm{avg}}(t)-c_s(t)$")
ax.text(0.02, 0.95, "(d)", transform=ax.transAxes, fontsize=8, fontweight="bold", va="top")
ax.legend(frameon=False, loc="best")

fig.savefig(FIGURES_DIR / "figure_05_linear_validation.pdf", format="pdf")

plt.show()
plt.close(fig)

In [ ]:
# Table 2: spatial/temporal convergence and active-constraint verification

import importlib.util

table_script = PROJECT_ROOT / "paper_tables" / "generate_tables.py"
spec = importlib.util.spec_from_file_location("generate_tables", table_script)
table_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(table_module)

table2 = table_module.compute_table2()
table2.to_csv(PROJECT_ROOT / "paper_tables" / "table_02_convergence.csv", index=False)
display(table2)
